# Carlos OS Aurora 0.10: construção no Colab

Gera a ISO no Colab e salva o resultado em `Meu Drive/Carlos-OS-Aurora`. Use **Ambiente de execução → Executar tudo**.

**Experimental:** este notebook nunca foi executado. Se o Colab bloquear `mount` ou `chroot`, a célula 3 para tudo, e a ISO não pode ser gerada aqui.

## 1. Conectar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/Carlos-OS-Aurora'

## 2. Baixar o projeto

In [ ]:
%cd /content
!test -d carlos-ia || git clone https://github.com/eduardoc7417-hub/carlos-ia.git
!git -C carlos-ia log --oneline -1

## 3. Verificar se o Colab permite a construção

In [ ]:
import subprocess

teste = r'''
nproc; free -h | head -2; df -h /opt | tail -1; id -u
mkdir -p /tmp/t-mount
mount -t tmpfs tmpfs /tmp/t-mount || exit 1
umount /tmp/t-mount
mkdir -p /tmp/t-proc
mount -t proc proc /tmp/t-proc || exit 2
umount /tmp/t-proc
chroot / /bin/true || exit 3
'''
r = subprocess.run(['bash', '-c', teste], capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode != 0:
    raise SystemExit('O Colab bloqueou mount ou chroot (codigo %d). A ISO nao pode ser gerada aqui.' % r.returncode)
print('Verificacao aprovada.')

## 4. Instalar as ferramentas

In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq --no-install-recommends live-build debootstrap debian-archive-keyring squashfs-tools xorriso \
  grub-pc-bin grub-efi-amd64-bin mtools dosfstools syslinux syslinux-common isolinux rsync python3-numpy python3-pil fonts-inter >/dev/null
lb --version

## 5. Gerar a ISO

O log fica em `Meu Drive/Carlos-OS-Aurora/build.log`. Se a sessão cair, rode tudo de novo: a construção recomeça do início.

In [ ]:
%%bash
cd /content/carlos-ia
D=/content/drive/MyDrive/Carlos-OS-Aurora
sed -i 's/carlos-os-aurora-0.9-/carlos-os-aurora-0.10-/' tools/build-iso.sh
sed -i 's/0\.9/0.10/g' config/live/hooks/normal/0100-carlos-identity.hook.chroot config/live/hooks/normal/0110-carlos-ui.hook.chroot
python3 tools/gen-art.py
sh tools/build-iso.sh > "$D/build.log" 2>&1
echo "build exit: $?"
tail -5 "$D/build.log" | cut -c1-160
grep -nE '^E: ' "$D/build.log" | tail -5 | cut -c1-200

## 6. Salvar a ISO no Drive

In [ ]:
%%bash
D=/content/drive/MyDrive/Carlos-OS-Aurora
cd /content/carlos-ia/out || exit 1
ls -lh
cp -v carlos-os-aurora-0.10-* "$D/"
sync
cd "$D" && sha256sum -c carlos-os-aurora-0.10-*.sha256